# Configurer la table cible du lakehouse et le watermark (version expliquée)

Associez `lh_meridian_hr` comme lakehouse par défaut, puis exécutez ce notebook comme **première** activité du pipeline d'ingestion des événements. Il réalise deux opérations idempotentes :

1. Précréer la table Delta partagée `bronze.workforce_events_raw` afin que les activités Copy parallèles ajoutent leurs données à une table déjà existante.
2. Créer `bronze.ingestion_watermark`, l'initialiser avec `default_watermark` uniquement la première fois, puis **renvoyer le watermark actuel** afin que l'étape suivante le reçoive comme paramètre.

Comme ce notebook prend en charge l'initialisation du watermark, le notebook de découverte en aval ne crée plus et ne lit plus la table de contrôle : il utilise simplement le watermark qui lui est transmis.

> Ceci est une copie annotée de `nb_setup_lakehouse.ipynb`. Chaque cellule de code est précédée d'un **Résumé** et d'une explication **ligne par ligne** repliable.

## Paramètres (marquer cette cellule comme cellule de paramètres)

**Résumé.** Identifie le pipeline dont cette exécution gère le watermark et la valeur à utiliser lors de la première initialisation.

<details>
<summary>Détails ligne par ligne</summary>

- `pipeline_name = "workforce_events"` — la clé qui identifie la ligne de ce pipeline dans `bronze.ingestion_watermark`.
- `default_watermark = "2020-12-01 00:00:00"` — initialisé uniquement lorsqu'aucune ligne n'existe encore; il s'agit volontairement d'un mois ancien afin que le premier chargement récupère tous les fichiers.

</details>

In [ ]:
pipeline_name = "workforce_events"
default_watermark = "2020-12-01 00:00:00"

## Créer le schéma et précréer la table cible

**Résumé.** Vérifie que le schéma `bronze` existe et précrée la table Delta vide `bronze.workforce_events_raw` avec une structure de colonnes explicite, afin que les activités Copy parallèles ajoutent leurs données à une table déjà existante au lieu de tenter de la créer simultanément.

<details>
<summary>Détails ligne par ligne</summary>

- `spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")` — crée le schéma `bronze` (base de données) s'il n'existe pas déjà; sans effet lors des réexécutions.
- `CREATE TABLE IF NOT EXISTS bronze.workforce_events_raw (...)` — définit la table d'événements bruts uniquement lorsqu'elle n'existe pas, ce qui rend le DDL idempotent (réutilisable sans risque).
- La liste de colonnes fixe le schéma en amont : identifiants (`event_id`, `employee_id`, `cost_center_id`), `event_date`, classification (`classification_group`, `classification_level`), `event_type`, montant (`amount_local` en `DECIMAL(18, 2)` et `local_currency`), `work_country_code`, ainsi que les colonnes de lignage (`ingest_ts`, `source_system`).
- `USING DELTA` — stocke la table comme une table Delta Lake (fichiers Parquet et journal des transactions), afin que les ajouts soient conformes à ACID.
- `print("bronze.workforce_events_raw is ready")` — écrit une ligne de confirmation dans la sortie de la cellule.

</details>

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.workforce_events_raw (
    event_id STRING,
    event_date DATE,
    employee_id STRING,
    cost_center_id STRING,
    classification_group STRING,
    classification_level INT,
    event_type STRING,
    amount_local DECIMAL(18, 2),
    local_currency STRING,
    work_country_code STRING,
    ingest_ts TIMESTAMP,
    source_system STRING
)
USING DELTA
""")

print("bronze.workforce_events_raw is ready")

## Créer, initialiser et renvoyer le watermark

**Résumé.** Crée la table de contrôle `bronze.ingestion_watermark` si elle est absente, initialise `default_watermark` uniquement lors de la première exécution, puis relit la valeur stockée et la renvoie afin que l'étape suivante du pipeline reçoive le watermark au lieu de le déduire.

<details>
<summary>Détails ligne par ligne</summary>

- Les deux contrôles rejettent un `pipeline_name` vide et un `default_watermark` qui ne correspond pas au premier jour d'un mois.
- `CREATE TABLE IF NOT EXISTS bronze.ingestion_watermark (...)` — crée la table de contrôle (`pipeline_name`, `watermark_timestamp`, `updated_at`) de manière idempotente.
- `spark.createDataFrame([...])` + `createOrReplaceTempView("watermark_seed")` — construit la source à une ligne utilisée par le MERGE.
- Le MERGE utilise uniquement **`WHEN NOT MATCHED`** : il insère la valeur par défaut lors de la première exécution et laisse le watermark existant inchangé lors des exécutions suivantes.
- La lecture `spark.table(...).where(col("pipeline_name") == pipeline_name)` renvoie la ligne qui existe désormais forcément : la valeur par défaut nouvellement initialisée lors de la première exécution, ou le watermark déjà stocké par la suite.
- `notebookutils.notebook.exit(json.dumps(result))` — renvoie `pipeline_name` et `watermark` au pipeline, qui transmet le watermark au notebook de découverte des fichiers.

</details>

In [ ]:
import json
from datetime import datetime

from pyspark.sql.functions import col

if not pipeline_name.strip():
    raise ValueError("pipeline_name must not be empty")

parsed_default = datetime.strptime(default_watermark, "%Y-%m-%d %H:%M:%S")
if parsed_default.day != 1:
    raise ValueError("default_watermark must be the first day of a month")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.ingestion_watermark (
    pipeline_name STRING,
    watermark_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

watermark_seed = spark.createDataFrame(
    [(pipeline_name, parsed_default)],
    "pipeline_name STRING, watermark_timestamp TIMESTAMP",
)
watermark_seed.createOrReplaceTempView("watermark_seed")

# Insert-only: an existing watermark is never overwritten.
spark.sql("""
MERGE INTO bronze.ingestion_watermark AS target
USING watermark_seed AS source
ON target.pipeline_name = source.pipeline_name
WHEN NOT MATCHED THEN INSERT (pipeline_name, watermark_timestamp, updated_at)
    VALUES (source.pipeline_name, source.watermark_timestamp, current_timestamp())
""")

watermark_row = (
    spark.table("bronze.ingestion_watermark")
    .where(col("pipeline_name") == pipeline_name)
    .select("watermark_timestamp")
    .collect()
)
watermark = watermark_row[0]["watermark_timestamp"].strftime("%Y-%m-%d %H:%M:%S")

result = {
    "pipeline_name": pipeline_name,
    "watermark": watermark,
}
print(result)
notebookutils.notebook.exit(json.dumps(result))